Importa las librerías necesarias: TensorFlow, NumPy y una función personalizada llamada `build` para construir el modelo, y `classification_report` de scikit-learn para evaluar el rendimiento del modelo.

In [1]:
import tensorflow as tf
import numpy as np
from TheModel import build
from sklearn.metrics import classification_report


### Carga y revisión de modelos locales

Este fragmento busca todos los archivos con extensión `.keras` en el directorio actual (y subdirectorios) y los carga como modelos de TensorFlow.  
Luego, verifica que **todos los modelos tengan la misma arquitectura**, comparando sus resúmenes (`model.summary()`).  
Si alguno es distinto, lanza un error para evitar inconsistencias en el entrenamiento federado.

In [2]:
## Abrir los modelos:

import os
loaded_local_models = [tf.keras.models.load_model(os.path.join(root, file)) for root, dirs, files in os.walk("./") for file in files if file.endswith('.keras')]

for i in range(len(loaded_local_models)-1):
    assert loaded_local_models[i].summary() == loaded_local_models[i+1].summary(), "Models have different architectures"

Model: "sequential_2"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 conv2d_4 (Conv2D)           (None, 26, 26, 32)        320       
                                                                 
 max_pooling2d_4 (MaxPooling  (None, 13, 13, 32)       0         
 2D)                                                             
                                                                 
 conv2d_5 (Conv2D)           (None, 11, 11, 64)        18496     
                                                                 
 max_pooling2d_5 (MaxPooling  (None, 5, 5, 64)         0         
 2D)                                                             
                                                                 
 flatten_2 (Flatten)         (None, 1600)              0         
                                                                 
 dense_4 (Dense)             (None, 64)               

###  Carga y preprocesamiento del dataset MNIST

Se carga el conjunto de datos MNIST desde TensorFlow.  
Luego, se normalizan las imágenes dividiendo entre 255 y se expanden las dimensiones para que tengan forma `(28, 28, 1)`, adecuada para modelos convolucionales.  
Finalmente, se asignan las etiquetas correspondientes para entrenamiento y prueba.

In [3]:
train, test = tf.keras.datasets.mnist.load_data()

x_train, x_test = np.expand_dims(train[0] / 255.0, -1), np.expand_dims(test[0] / 255.0, -1)
y_train, y_test = train[1], test[1]

###  Average Trimmed  
Aplicación del modelo de aprendizaje federado utilizando un promedio recortado de los pesos. Este método descarta un porcentaje de los valores más altos y más bajos de cada capa antes de promediarlos, lo que ayuda a reducir el impacto de valores extremos o anómalos provenientes de clientes particulares. El modelo resultante se evalúa con los datos de prueba y se guarda como `trimmed_model.keras`.


In [8]:
def FedTrimmedMean(weights, trim_ratio):
    n_clients = len(weights)
    trim = int(trim_ratio * n_clients)
    trimmed_weights = []
    for layer_weights in zip(*weights):
        sorted_weights = np.sort(layer_weights, axis=0)
        trimmed_layer_weights = sorted_weights[trim:n_clients-trim]
        trimmed_mean = np.mean(trimmed_layer_weights, axis=0)
        trimmed_weights.append(trimmed_mean)
    return trimmed_weights

local_weights = [x.get_weights() for x in loaded_local_models]
averaged_weights = FedTrimmedMean(local_weights, trim_ratio=0.1)

trimmed_model = build.build_it()
trimmed_model.set_weights(averaged_weights)

from sklearn.metrics import classification_report
# Predict the classes for the test set
y_pred = trimmed_model.predict(x_test)
y_pred_classes = np.argmax(y_pred, axis=1)

print(classification_report(y_test, y_pred_classes))

trimmed_model.save('trimmed_model.keras')

313/313 [==============================] - 2s 7ms/step
              precision    recall  f1-score   support

           0       0.92      0.14      0.25       980
           1       0.62      0.94      0.75      1135
           2       0.98      0.08      0.15      1032
           3       0.93      0.36      0.52      1010
           4       0.24      0.98      0.38       982
           5       0.77      0.45      0.57       892
           6       1.00      0.02      0.04       958
           7       0.84      0.55      0.67      1028
           8       0.52      0.89      0.66       974
           9       0.26      0.19      0.22      1009

    accuracy                           0.47     10000
   macro avg       0.71      0.46      0.42     10000
weighted avg       0.71      0.47      0.42     10000

